# Studio: gramática de conversación visible · 2.1.4

System conversacional mínimo con los tres SDKs y Python. Input mensaje/historia → Agent y Tool → texto e historia observada. No certifica todos los controles de la interfaz. El servidor real se abre en el perfil local; instalación, controlador completo y UI tienen pruebas separadas.

In [2]:
import pydantic

In [3]:
import agentic_systems as toolkit

In [4]:
class Message(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra="forbid")
    role: str
    content: str

In [5]:
class Conversation(pydantic.BaseModel):
    message: str
    history: list[Message] = pydantic.Field(default_factory=list)
    text: str | None = None
    history_turns: int | None = None

In [6]:
@toolkit.tool
def reply(message: str, history: list[Message]) -> dict:
    """A deterministic conversation fixture, not an LM response."""
    return {"text": "Python received: " + message, "history_turns": len(history)}

In [7]:
def build(framework):
    actor = toolkit.Agent(
        "conversation_actor",
        runtime=toolkit.runtime(provider="python-runtime"),
        framework=toolkit.framework(framework),
        skills=(toolkit.skill(name="conversation", tools=(reply,)),),
    )
    graph = toolkit.Graph(Conversation)
    graph.add_node("respond", actor)
    graph.add_edge(toolkit.START, "respond")
    graph.add_edge("respond", toolkit.END)
    return toolkit.System(
        "studio_conversation_grammar",
        graph=graph,
        context=toolkit.Context(
            {"history_policy": "explicit input, no inferred facts"}
        ),
        memory=toolkit.Memory(capacity=3),
    )

In [8]:
def run():
    rows = []
    for framework in ("native", "openai-agents", "strands"):
        app = build(framework)
        first = app.run({"message": "Hola", "history": []})
        assert first.ok, first.to_dict()
        second = app.run(
            {
                "message": "Continua",
                "history": [
                    {"role": "user", "content": "Hola"},
                    {"role": "assistant", "content": first.data["text"]},
                ],
            }
        )
        assert second.ok and second.data["history_turns"] == 2, second.to_dict()
        rows.append(
            {
                "framework": framework,
                "first": first.to_dict(),
                "second": second.to_dict(),
            }
        )
    return {
        "rows": rows,
        "scope": "visible SDK conversation composition; full Studio controller/UI tested separately",
        "inference_requests": 0,
    }

In [9]:
report = run()
result = toolkit.RunResult(**report["rows"][0]["second"])
print(result.human_results())

studio_conversation_grammar: COMPLETED
Execution: run-941f587ecad04bf3b6bf5aafedd4f2aa
Input: {"message":"Continua","history":[{"role":"user","content":"Hola"},{"role":"assistant","content":"Python received: Hola"}]}
Output: {"message":"Continua","history":[{"role":"user","content":"Hola"},{"role":"assistant","content":"Python received: Hola"}],"text":"Python received: Continua","history_turns":2}
Process:
- Agent conversation_actor: completed
- Tool reply: completed -> {"text":"Python received: Continua","history_turns":2}


## Interfaz real: perfil local separado

Requiere instalar Studio. Este bloque comprueba inicio/parada, no todos los controles UI.

In [ ]:
from agentic_systems_studio.server import start_studio_server

server = start_studio_server(
    port=8503, log_dir=".tmp/studio-dev6-notebook", stop_previous=False
)
try:
    print(server.local_url)
finally:
    server.stop()